# 14 - Foursquare Feasibility

This notebook tests whether Foursquare Places can improve naming and identity for the most important Istanbul POIs.

Purpose:
- query Foursquare only for the top important POIs
- compare returned venue names with current OSM / `name_en` values
- decide whether Foursquare is worth using as a naming-enrichment layer

Important:
- This notebook is **not** for replacing OSM.
- It is only for testing whether Foursquare improves naming / identity for important landmarks.

In [31]:
import os
import time
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load enriched POIs

In [32]:
pois = pd.read_csv("../data/processed/poi_enriched.csv")
pois.head()

,poi_id,name,name_en,category,category_clean,lat,lon,distance_to_center_km,nearby_count_500m,cluster_id,category_score,landmark_name_score,centrality_score,density_score,importance_score,is_park,is_historic,is_museum,is_attraction,is_religious
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007586,28.975554,0.248384,64,12,0.82,0.55,0.997367,0.941176,0.863004,0,1,0,0,0
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007289,28.975329,0.276907,63,12,0.82,0.55,0.997021,0.926471,0.859224,0,1,0,0,0
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic:castle,historic,41.006397,28.974808,0.361956,63,12,0.82,0.55,0.995990,0.926471,0.858915,0,1,0,0,0
3,311681431,Sağlık Müzesi,NaN,tourism:museum,museum,41.008314,28.975290,0.261290,66,12,0.85,0.35,0.997210,0.970588,0.849310,0,0,1,0,0
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,tourism:museum,museum,41.004295,28.977433,0.441766,55,12,0.85,0.59,0.995023,0.808824,0.844213,0,0,1,0,0


## Select the important landmark subset

We keep this small so Foursquare remains a quick go/no-go test.

In [33]:
TARGET_CATEGORIES = ["museum", "historic", "attraction"]
TOP_N = 5

test_pois = (
    pois[pois["category_clean"].isin(TARGET_CATEGORIES)]
    .sort_values("importance_score", ascending=False)
    .head(TOP_N)
    .copy()
)

test_pois["match_name"] = test_pois["name_en"].fillna(test_pois["name"])
test_pois[["poi_id", "name", "name_en", "match_name", "category_clean", "importance_score"]].head(20)

,poi_id,name,name_en,match_name,category_clean,importance_score
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,historic,0.863004
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,historic,0.859224
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,historic,0.858915
3,311681431,Sağlık Müzesi,NaN,Sağlık Müzesi,museum,0.849310
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,museum,0.844213


In [34]:
import os
print("FOURSQUARE_API_KEY" in os.environ)
print(os.getenv("FOURSQUARE_API_KEY"))


False
None


## Foursquare configuration

According to the official Foursquare Places API docs:
- authentication uses the `Authorization` header with the API key
- place search endpoint is `https://api.foursquare.com/v3/places/search`
- the required version header is `X-Places-Api-Version: 1970-01-01`

Set your key in the environment before running:

```bash
export FOURSQUARE_API_KEY="..."
```

In [35]:
import os

FOURSQUARE_API_KEY = os.getenv("FOURSQUARE_API_KEY")

if not FOURSQUARE_API_KEY:
    FOURSQUARE_API_KEY = input("Paste Foursquare API key: ").strip()

FSQ_SEARCH_URL = "https://places-api.foursquare.com/places/search"

print("Key loaded:", bool(FOURSQUARE_API_KEY))


Key loaded: True


## Helper function

In [36]:
def search_foursquare_place(query, lat, lon, limit=5, max_retries=5):
    if not FOURSQUARE_API_KEY:
        raise ValueError("FOURSQUARE_API_KEY is missing")

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {FOURSQUARE_API_KEY}",
        "X-Places-Api-Version": "2025-06-17",
    }

    params = {
        "query": query,
        "ll": f"{lat},{lon}",
        "radius": 500,
        "limit": limit,
        "sort": "RELEVANCE",
        "fields": "fsq_place_id,name,categories,location,rating,distance",
    }

    last_debug = {}

    for attempt in range(max_retries):
        response = requests.get(FSQ_SEARCH_URL, headers=headers, params=params, timeout=20)

        if response.status_code == 200:
            payload = response.json()
            if last_debug:
                payload['_rate_limit_debug'] = last_debug
            return payload

        if response.status_code == 429:
            header_names = [
                "Retry-After",
                "X-RateLimit-Limit",
                "X-RateLimit-Remaining",
                "X-RateLimit-Reset",
                "X-Factual-Throttle-Allocation",
            ]
            debug = {name: response.headers.get(name) for name in header_names}
            debug['status_code'] = response.status_code
            debug['response_text'] = response.text[:300]
            last_debug = debug

            retry_after = response.headers.get('Retry-After')
            if retry_after:
                try:
                    wait_time = max(float(retry_after), 1.0)
                except ValueError:
                    wait_time = float(2 ** attempt)
            else:
                wait_time = float(2 ** attempt)

            print(f"Rate limited for '{query}' on attempt {attempt + 1}/{max_retries}.")
            print("429 headers:", debug)
            print(f"Waiting {wait_time:.1f}s before retry...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()

    raise Exception(f"Foursquare search failed after retries for query: {query}. Last debug: {last_debug}")


## Test a few landmark queries first

In [37]:
sample_queries = test_pois[["match_name", "lat", "lon"]].head(5).to_dict(orient="records")

for row in sample_queries:
    print(f"\n===== {row['match_name']} =====")
    if not FOURSQUARE_API_KEY:
        print("Skipping because FOURSQUARE_API_KEY is missing")
        continue
    try:
        result = search_foursquare_place(row["match_name"], row["lat"], row["lon"], limit=3)
        for item in result.get("results", []):
            print(item.get("name"), "|", item.get("fsq_place_id"), "|", item.get("distance"))
    except Exception as e:
        print("ERROR:", e)


===== Lausos Sarayı'nın Kalıntıları =====
Rate limited for 'Lausos Sarayı'nın Kalıntıları' on attempt 1/5.
429 headers: {'Retry-After': None, 'X-RateLimit-Limit': '0', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': None, 'X-Factual-Throttle-Allocation': None, 'status_code': 429, 'response_text': '{"message":"Your account has no API credits remaining. Please visit your organization’s billing page at https://foursquare.com/developers/orgs to manually add credits or enable automatic payments. Purchasing credits is required if you are trying to make Premium calls or you have exceeded your freePr'}
Waiting 1.0s before retry...
Rate limited for 'Lausos Sarayı'nın Kalıntıları' on attempt 2/5.
429 headers: {'Retry-After': None, 'X-RateLimit-Limit': '0', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': None, 'X-Factual-Throttle-Allocation': None, 'status_code': 429, 'response_text': '{"message":"Your account has no API credits remaining. Please visit your organization’s billing page at h

## Run the top-POI Foursquare name test

In [38]:
fsq_rows = []

for _, row in test_pois.iterrows():
    query = row["match_name"]

    record = {
        "poi_id": row["poi_id"],
        "osm_name": row["name"],
        "osm_name_en": row["name_en"],
        "match_name": query,
        "category_clean": row["category_clean"],
        "importance_score": row["importance_score"],
        "foursquare_found": False,
        "foursquare_name": None,
        "foursquare_id": None,
        "foursquare_distance_m": None,
        "foursquare_category": None,
        "rate_limit_retry_after": None,
        "rate_limit_remaining": None,
        "rate_limit_reset": None,
        "rate_limit_allocation": None,
        "notes": None,
    }

    if not FOURSQUARE_API_KEY:
        fsq_rows.append(record)
        continue

    try:
        result = search_foursquare_place(query, row["lat"], row["lon"], limit=5)
        places = result.get("results", [])
        debug = result.get('_rate_limit_debug', {})
        record.update({
            "rate_limit_retry_after": debug.get('Retry-After'),
            "rate_limit_remaining": debug.get('X-RateLimit-Remaining'),
            "rate_limit_reset": debug.get('X-RateLimit-Reset'),
            "rate_limit_allocation": debug.get('X-Factual-Throttle-Allocation'),
        })

        if places:
            top = places[0]
            cats = top.get("categories", [])
            record.update({
                "foursquare_found": True,
                "foursquare_name": top.get("name"),
                "foursquare_id": top.get("fsq_place_id"),
                "foursquare_distance_m": top.get("distance"),
                "foursquare_category": cats[0].get("name") if cats else None,
            })
    except Exception as e:
        record["notes"] = str(e)

    fsq_rows.append(record)
    time.sleep(2.0)

foursquare_results = pd.DataFrame(fsq_rows)
foursquare_results.head(20)


Rate limited for 'Lausos Sarayı'nın Kalıntıları' on attempt 1/5.
429 headers: {'Retry-After': None, 'X-RateLimit-Limit': '0', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': None, 'X-Factual-Throttle-Allocation': None, 'status_code': 429, 'response_text': '{"message":"Your account has no API credits remaining. Please visit your organization’s billing page at https://foursquare.com/developers/orgs to manually add credits or enable automatic payments. Purchasing credits is required if you are trying to make Premium calls or you have exceeded your freePr'}
Waiting 1.0s before retry...
Rate limited for 'Lausos Sarayı'nın Kalıntıları' on attempt 2/5.
429 headers: {'Retry-After': None, 'X-RateLimit-Limit': '0', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': None, 'X-Factual-Throttle-Allocation': None, 'status_code': 429, 'response_text': '{"message":"Your account has no API credits remaining. Please visit your organization’s billing page at https://foursquare.com/developers/orgs to ma

,poi_id,osm_name,osm_name_en,match_name,category_clean,importance_score,foursquare_found,foursquare_name,foursquare_id,foursquare_distance_m,foursquare_category,rate_limit_retry_after,rate_limit_remaining,rate_limit_reset,rate_limit_allocation,notes
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,historic,0.863004,False,None,None,None,None,None,None,None,None,Foursquare search failed after retries for que...
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,historic,0.859224,False,None,None,None,None,None,None,None,None,Foursquare search failed after retries for que...
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,historic,0.858915,False,None,None,None,None,None,None,None,None,Foursquare search failed after retries for que...
3,311681431,Sağlık Müzesi,NaN,Sağlık Müzesi,museum,0.849310,False,None,None,None,None,None,None,None,None,Foursquare search failed after retries for que...
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,museum,0.844213,False,None,None,None,None,None,None,None,None,Foursquare search failed after retries for que...


## Manual evaluation fields

These columns make the go/no-go decision easy.

In [39]:
foursquare_results["clearly_better_name"] = ""
foursquare_results["would_use_for_wiki"] = ""
foursquare_results["would_use_for_besttime"] = ""

foursquare_results[[
    "poi_id",
    "osm_name",
    "osm_name_en",
    "match_name",
    "foursquare_found",
    "foursquare_name",
    "foursquare_distance_m",
    "foursquare_category",
    "clearly_better_name",
    "would_use_for_wiki",
    "would_use_for_besttime",
    "notes",
]].head(30)

,poi_id,osm_name,osm_name_en,match_name,foursquare_found,foursquare_name,foursquare_distance_m,foursquare_category,clearly_better_name,would_use_for_wiki,would_use_for_besttime,notes
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,False,None,None,None,,,,Foursquare search failed after retries for que...
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,False,None,None,None,,,,Foursquare search failed after retries for que...
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,False,None,None,None,,,,Foursquare search failed after retries for que...
3,311681431,Sağlık Müzesi,NaN,Sağlık Müzesi,False,None,None,None,,,,Foursquare search failed after retries for que...
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,False,None,None,None,,,,Foursquare search failed after retries for que...


## Save results for review

In [40]:
if not FOURSQUARE_API_KEY:
    print("Skipping save because no Foursquare API key was provided.")
else:
    foursquare_results.to_csv("../data/processed/foursquare_name_test_top30.csv", index=False)
    print("Saved: ../data/processed/foursquare_name_test_top30.csv")
    print("Shape:", foursquare_results.shape)

Saved: ../data/processed/foursquare_name_test_top30.csv
Shape: (5, 19)


## Decision rule

Use Foursquare only if:
- setup is easy,
- names are clearly better for a majority of the tested landmarks,
- and it directly improves Wikipedia / BestTime query naming.

Otherwise, keep OSM + manually cleaned English names and move on.